In [ ]:
import math
import sys
from datetime import datetime
from pathlib import Path

import torch

# ==================== 1. Environment and Path Configuration ====================
base_dir = Path(__file__).parent if '__file__' in locals() else Path.cwd()
project_root = base_dir.parent if base_dir.name == 'simclr_finetune' else base_dir

# Ensure project root is in Python module search path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import simclr_finetune modules
from finetune.lib import (
    load_pretrained_encoder,
    save_model_checkpoint,
    BinaryClassifier,
    prepare_finetune_dataset,
    train_binary_classifier,
    evaluate_and_record_predictions,
    evaluate_positive_per_smiles,
    export_results_to_excel,
    plot_comprehensive_results,
    plot_confusion_matrices
)

# Hyperparameter configuration
batch_size = 128
lr = 0.001
epochs = 100
patience = 15
freeze_encoder = True
pos_weight = math.sqrt(1.8)  # Positive class loss weighting pos_weight = sqrt(1.8) ≈ 1.34164
threshold = 0.5  # Inference decision threshold: 0.5

# Set inference and fine-tuning device (CUDA preferred)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 60, flush=True)
print("SimCLR Mass Spectrum Binary Classification Fine-Tuning Pipeline", flush=True)
print("=" * 60, flush=True)
print(f"Device: {device.upper()}", flush=True)
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Inference threshold: {threshold}", flush=True)

# Data and model file paths
pos_msp_path = base_dir / "data_source" / "positive_cannabinoids.msp"
neg_msp_path = base_dir / "data_source" / "negative.msp"
encoder_path = project_root / "embedding_pretrain" / "pretrained_encoder_final_v1.pt"

print(f"Positive sample path: {pos_msp_path}", flush=True)
print(f"Negative sample path: {neg_msp_path}", flush=True)
print(f"Encoder path: {encoder_path}", flush=True)

# ==================== 2. Data Preparation and Splitting ====================
print("\n" + "=" * 60, flush=True)
print("Step 1: Data loading, preprocessing, and splitting", flush=True)
print("=" * 60, flush=True)
train_loader, train_eval_loader, val_loader, test_loader, meta = prepare_finetune_dataset(
    pos_msp_path=pos_msp_path,
    neg_msp_path=neg_msp_path,
    batch_size=batch_size,
    test_size=0.15,
    val_size=0.15,
    random_state=42,
    use_cuda=(device == 'cuda')
)

print(f"  Training batches: {len(train_loader)}", flush=True)
print(f"  Validation batches: {len(val_loader)}", flush=True)
print(f"  Test batches: {len(test_loader)}", flush=True)

# ==================== 3. Model Construction and Weight Loading ====================
print("\n" + "=" * 60, flush=True)
print("Step 2: Building model and loading pre-trained encoder", flush=True)
print("=" * 60, flush=True)
encoder = load_pretrained_encoder(
    encoder_path=encoder_path,
    input_dim=561,
    hidden_dim=256,
    device=device
)
model = BinaryClassifier(encoder, input_dim=256, freeze_encoder=freeze_encoder)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable_params:,}", flush=True)

# ==================== 4. Fine-Tuning Training ====================
print("\n" + "=" * 60, flush=True)
print("Step 3: Starting binary classification fine-tuning", flush=True)
print("=" * 60, flush=True)
start_time = datetime.now()
model, history = train_binary_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=epochs,
    lr=lr,
    patience=patience,
    pos_weight=pos_weight
)
elapsed = datetime.now() - start_time
print(f"  [OK] Training complete, total time: {str(elapsed).split('.')[0]}", flush=True)

# ==================== 5. Comprehensive Evaluation ====================
print("\n" + "=" * 60, flush=True)
print(f"Step 4: Model performance evaluation (threshold={threshold})", flush=True)
print("=" * 60, flush=True)
train_res = evaluate_and_record_predictions(model, train_eval_loader, sample_names=meta['train_sample_names'], device=device, threshold=threshold)
val_res = evaluate_and_record_predictions(model, val_loader, sample_names=meta['val_sample_names'], device=device, threshold=threshold)
test_res = evaluate_and_record_predictions(model, test_loader, sample_names=meta['test_sample_names'], device=device, threshold=threshold)

print(f"  Training Set | Accuracy: {train_res['accuracy']:.2%} | AUC: {train_res['auc']:.4f}", flush=True)
print(f"  Validation Set | Accuracy: {val_res['accuracy']:.2%} | AUC: {val_res['auc']:.4f}", flush=True)
print(f"  Test Set | Accuracy: {test_res['accuracy']:.2%} | AUC: {test_res['auc']:.4f}", flush=True)

smiles_res = evaluate_positive_per_smiles(
    test_indices=meta['test_idx'],
    smiles_all=meta['smiles_all'],
    labels_all=meta['y'],
    probs=test_res['probs'],
    preds=test_res['preds'],
    threshold=threshold
)

In [ ]:
# ==================== 6. Results Saving and Export ====================
print("\n" + "=" * 60, flush=True)
print("Step 5: Saving model and exporting evaluation results", flush=True)
print("=" * 60, flush=True)
# Create a unified timestamped output directory for this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = base_dir / f"results_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)

# 1. Save model weights to the dedicated results directory
print("  [DEBUG] Calling save_model_checkpoint...", flush=True)
saved_path = save_model_checkpoint({
    'encoder_state_dict': encoder.state_dict(),
    'classifier_state_dict': model.classifier.state_dict(),
    'history': history,
    'test_metrics': test_res,
}, run_dir)

# 2. Export CSV / Excel evaluation detail tables to the dedicated results directory
print("  [DEBUG] Calling export_results_to_excel...", flush=True)
export_results_to_excel(
    history=history,
    model=model,
    train_results=train_res,
    val_results=val_res,
    test_results=test_res,
    smiles_results=smiles_res,
    meta=meta,
    output_dir=run_dir
)

# 3. Save all visualization charts to the dedicated results directory
print("  [DEBUG] Calling plot_comprehensive_results...", flush=True)
plot_comprehensive_results(history, train_res, val_res, test_res, smiles_res, run_dir)
print("  [DEBUG] Calling plot_confusion_matrices...", flush=True)
plot_confusion_matrices(train_res, val_res, test_res, run_dir)

print(f"\n  [OK] All output results (model weights .pt, evaluation tables CSV, and charts PNG) have been saved to:", flush=True)
print(f"       --> {run_dir}", flush=True)
print("\n" + "=" * 60, flush=True)
print("Fine-tuning pipeline completed successfully!", flush=True)
print("=" * 60, flush=True)